> **Jak se učit:** Přečtěte si zadání každého cvičení, zkuste navrhnout postup, poté projděte řešení s komentáři. Buňky spouštějte postupně (`Shift+Enter`). Logovací cvičení vytvářejí soubory `.log` ve složce, ze které Jupyter běží.

# Logování — monitoring aplikací v Pythonu

Základem vývoje zdravé aplikace je správné nastavení **monitoringu**. Jeho základem je pro nás **logování** — systematické zaznamenávání událostí, které se v aplikaci dějí.

---

## Proč logovat?

| Důvod | Příklad |
|---|---|
| **Hledání chyb** | Díky logům zjistíme, co se stalo těsně před pádem aplikace |
| **Analýza využívání** | Kolik uživatelů používá danou funkci? |
| **Kontrola pohybu dat** | Někdy vyžadována zákony (GDPR, státní regulace) |
| **Diagnostika výkonu** | Jak dlouho trvá zpracování požadavku? |
| **Bezpečnost** | Kdo se pokusil přihlásit? Z jaké IP? |

## Hlavní části logovacího procesu

| Část | Zodpovědnost |
|---|---|
| **Logger** | Objekt, přes který v kódu **zapisujeme** zprávy |
| **Handler** | Určuje **kam** se zprávy posílají (soubor, konzole, síť…) |
| **Formatter** | Určuje **formát** zprávy (čas, úroveň, text…) |
| **Filter** | Umožňuje **filtrovat**, které zprávy se zaznamenají |

```
Logger  ──►  Filter  ──►  Handler  ──►  Formatter  ──►  Výstup (soubor/konzole/...)
```

## Úrovně závažnosti (log levels)

Každá zpráva má svou **úroveň závažnosti** — od nejméně po nejvíce kritickou:

| Úroveň | Číslo | Kdy použít | Příklad |
|---|---|---|---|
| `DEBUG` | 10 | Detailní informace pro vývojáře | `Proměnná x = 42` |
| `INFO` | 20 | Potvrzení, že vše funguje správně | `Server spuštěn na portu 8080` |
| `WARNING` | 30 | Něco neočekávaného, ale aplikace běží | `Disk je zaplněn na 90 %` |
| `ERROR` | 40 | Chyba — něco se nezdařilo | `Nepodařilo se připojit k DB` |
| `CRITICAL` | 50 | Fatální chyba — aplikace nemůže pokračovat | `Systém spadl, data ztracena` |

> **Tip:** Logger zobrazí pouze zprávy na své úrovni **a výš**. Logger s `level=WARNING` zobrazí WARNING, ERROR a CRITICAL, ale ne DEBUG a INFO.

## Základní pravidla logování

1. **Používat logovací knihovnu** — v Pythonu vestavěný modul `logging`, nikdy ne `print()`
2. **Logovat na správné úrovni** — ulehčuje filtrování a analýzu
3. **Psát užitečné zprávy** — co se stalo, s jakými hodnotami, pro koho
4. **Používat angličtinu** — zamezuje problémům s kódováním (ASCII) a jazykovou bariérou
5. **Nelogovat citlivé údaje** — hesla, tokeny, osobní údaje
6. **Logovat do více cílů** — konzole pro vývoj, soubor pro produkci

---

# Modul `logging` — základní použití

Python má vestavěný modul `logging` — **nemusíme nic instalovat**.

## Nejjednodušší použití — `basicConfig`

Funkce `logging.basicConfig()` nastaví výchozí logger, handler i formatter jedním voláním:

In [1]:
import logging

# Nastavení základního logování — výstup do konzole
logging.basicConfig(
    level=logging.DEBUG,  # minimální úroveň zpráv
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Zkusíme všechny úrovně
logging.debug("Toto je DEBUG zpráva")
logging.info("Toto je INFO zpráva")
logging.warning("Toto je WARNING zpráva")
logging.error("Toto je ERROR zpráva")
logging.critical("Toto je CRITICAL zpráva")

2026-10-18 22:26:32,880 - DEBUG - Toto je DEBUG zpráva


2026-10-18 22:26:32,880 - INFO - Toto je INFO zpráva


2026-10-18 22:26:32,881 - WARNING - Toto je WARNING zpráva


2026-10-18 22:26:32,881 - ERROR - Toto je ERROR zpráva


2026-10-18 22:26:32,882 - CRITICAL - Toto je CRITICAL zpráva


> **⚠️ Pozor:** `basicConfig()` funguje pouze jednou (při prvním volání). V Jupyter Notebooku se může stát, že opakované spouštění buněk neprovede změny. V takovém případě restartujte kernel.

> **Pozn.:** Výchozí úroveň (bez `basicConfig`) je `WARNING` — zprávy `DEBUG` a `INFO` se **nezobrazí**.

---

# Logger — vytváření pojmenovaných loggerů

V praxi nepoužíváme přímo `logging.info(...)`, ale vytvoříme si **pojmenovaný logger**. To nám umožní:

- Rozlišit, **z jaké části** aplikace zpráva pochází
- Nastavit **různé úrovně** pro různé části
- Přidat **různé handlery** (soubor, konzole…)

## Syntaxe

```python
logger = logging.getLogger("nazev_loggeru")
```

> **Tip:** Jako název se často používá `__name__` — automaticky se doplní název modulu/souboru.

## Hierarchie loggerů

Loggery tvoří **stromovou hierarchii** oddělenou tečkami. Kořenový (root) logger je nadřazený všem:

```
root
├── app
│   ├── app.database
│   └── app.auth
└── utils
```

Zprávy z podřízeného loggeru **probublají nahoru** k nadřazenému (pokud to nezakážeme).

### Příklad

In [2]:
import logging

# Kořenový logger
root_logger = logging.getLogger()

# Pojmenované loggery v hierarchii
app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")
auth_logger = logging.getLogger("app.auth")

# Přidáme handler na root logger (všechny zprávy se sem dostanou)
console = logging.StreamHandler()
console.setFormatter(logging.Formatter("%(name)s - %(levelname)s - %(message)s"))
root_logger.addHandler(console)
root_logger.setLevel(logging.DEBUG)

# Zprávy z různých loggerů
root_logger.info("Zpráva z root loggeru")
app_logger.info("Zpráva z app loggeru")
db_logger.warning("Zpráva z app.database loggeru")
auth_logger.error("Zpráva z app.auth loggeru")

# ☝️ Všechny zprávy se zobrazí, protože probublají do root loggeru

2026-10-18 22:26:32,886 - INFO - Zpráva z root loggeru


root - INFO - Zpráva z root loggeru


2026-10-18 22:26:32,887 - INFO - Zpráva z app loggeru


app - INFO - Zpráva z app loggeru


2026-10-18 22:26:32,887 - WARNING - Zpráva z app.database loggeru


app.database - WARNING - Zpráva z app.database loggeru


2026-10-18 22:26:32,888 - ERROR - Zpráva z app.auth loggeru


app.auth - ERROR - Zpráva z app.auth loggeru


---

# Handler — kam se zprávy posílají

Handler určuje **destinaci** logů. Jeden logger může mít **více handlerů** (např. současně konzole + soubor).

## Nejčastější typy handlerů

| Handler | Kam zapisuje | Import |
|---|---|---|
| `StreamHandler` | Konzole (stdout/stderr) | `logging.StreamHandler()` |
| `FileHandler` | Soubor | `logging.FileHandler('app.log')` |
| `RotatingFileHandler` | Soubor s **rotací** (max. velikost) | `logging.handlers.RotatingFileHandler(...)` |
| `TimedRotatingFileHandler` | Soubor s **časovou rotací** (denně, týdně…) | `logging.handlers.TimedRotatingFileHandler(...)` |

> **Tip:** Rotace souborů je důležitá v produkci — bez ní by log soubor neomezeně rostl.

### Příklad — StreamHandler (konzole)

In [3]:
import logging

logger = logging.getLogger("ukazka_stream")
logger.setLevel(logging.DEBUG)

# Handler pro výstup do konzole
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)

# Přiřadíme handler k loggeru
logger.addHandler(console_handler)

logger.info("Tato zpráva se zobrazí v konzoli")
logger.debug("A tato také")

Tato zpráva se zobrazí v konzoli


2026-10-18 22:26:32,892 - INFO - Tato zpráva se zobrazí v konzoli


ukazka_stream - INFO - Tato zpráva se zobrazí v konzoli


A tato také


2026-10-18 22:26:32,893 - DEBUG - A tato také


ukazka_stream - DEBUG - A tato také


### Příklad — FileHandler (soubor)

In [4]:
import logging

logger = logging.getLogger("ukazka_file")
logger.setLevel(logging.DEBUG)

# Handler pro zápis do souboru
# mode="w" → přepíše soubor při každém spuštění
# mode="a" → připojí na konec (výchozí)
file_handler = logging.FileHandler("ukazka.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.INFO)  # do souboru zapíše jen INFO a výš

logger.addHandler(file_handler)

logger.debug("DEBUG — toto se do souboru NEZAPÍŠE (handler má level INFO)")
logger.info("INFO — toto se do souboru zapíše")
logger.warning("WARNING — toto se do souboru zapíše")

print("✅ Zkontrolujte soubor 'ukazka.log' ve stejné složce.")

2026-10-18 22:26:32,897 - DEBUG - DEBUG — toto se do souboru NEZAPÍŠE (handler má level INFO)


ukazka_file - DEBUG - DEBUG — toto se do souboru NEZAPÍŠE (handler má level INFO)


2026-10-18 22:26:32,898 - INFO - INFO — toto se do souboru zapíše


ukazka_file - INFO - INFO — toto se do souboru zapíše


2026-10-18 22:26:32,899 - WARNING - WARNING — toto se do souboru zapíše


ukazka_file - WARNING - WARNING — toto se do souboru zapíše


✅ Zkontrolujte soubor 'ukazka.log' ve stejné složce.


---

# Formatter — formát zprávy

Formatter určuje, **jak bude zpráva vypadat**. Definujeme ho jako textový vzor s proměnnými.

## Nejčastější proměnné formátovacího řetězce

| Proměnná | Popis | Příklad výstupu |
|---|---|---|
| `%(asctime)s` | Datum a čas | `2025-10-10 09:15:21,362` |
| `%(name)s` | Název loggeru | `app.database` |
| `%(levelname)s` | Úroveň zprávy | `WARNING` |
| `%(message)s` | Text zprávy | `Disk plný` |
| `%(filename)s` | Název souboru | `main.py` |
| `%(funcName)s` | Název funkce | `process_data` |
| `%(lineno)d` | Číslo řádku | `42` |
| `%(module)s` | Název modulu | `main` |
| `%(process)d` | ID procesu | `12345` |
| `%(thread)d` | ID vlákna | `140735` |

### Příklad

In [5]:
import logging

logger = logging.getLogger("ukazka_formatter")
logger.setLevel(logging.DEBUG)

# Handler s formatterem
handler = logging.StreamHandler()
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)-8s | %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S"  # vlastní formát data
)
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.debug("Detailní zpráva")
logger.info("Vše běží správně")
logger.warning("Pozor, něco je podezřelé")
logger.error("Nastala chyba")

18-10-2026 22:26:32 | ukazka_formatter | DEBUG    | Detailní zpráva


2026-10-18 22:26:32,904 - DEBUG - Detailní zpráva


ukazka_formatter - DEBUG - Detailní zpráva


18-10-2026 22:26:32 | ukazka_formatter | INFO     | Vše běží správně


2026-10-18 22:26:32,906 - INFO - Vše běží správně


...

2026-10-18 22:26:32,908 - WARNING - Pozor, něco je podezřelé


ukazka_formatter - WARNING - Pozor, něco je podezřelé


18-10-2026 22:26:32 | ukazka_formatter | ERROR    | Nastala chyba


2026-10-18 22:26:32,909 - ERROR - Nastala chyba


ukazka_formatter - ERROR - Nastala chyba


---

# Kombinace — konzole + soubor

V praxi chceme obvykle logovat **současně do konzole i do souboru**, často s **různými úrovněmi**:

- **Konzole** → `DEBUG` a výše (pro vývojáře při vývoji)
- **Soubor** → `WARNING` a výše (pro produkční analýzu)

### Příklad

In [6]:
import logging

logger = logging.getLogger("moje_aplikace")
logger.setLevel(logging.DEBUG)  # logger propouští vše od DEBUG

# Handler 1: konzole — zobrazí vše (DEBUG+)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_formatter = logging.Formatter("%(levelname)-8s | %(message)s")
console_handler.setFormatter(console_formatter)

# Handler 2: soubor — zapíše jen WARNING+
file_handler = logging.FileHandler("aplikace.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.WARNING)
file_formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(file_formatter)

# Přidáme oba handlery
logger.addHandler(console_handler)
logger.addHandler(file_handler)

# Testovací zprávy
logger.debug("Debug — pouze konzole")
logger.info("Info — pouze konzole")
logger.warning("Warning — konzole + soubor")
logger.error("Error — konzole + soubor")

print("\n✅ Zkontrolujte soubor 'aplikace.log' — najdete tam jen WARNING a ERROR.")

DEBUG    | Debug — pouze konzole


2026-10-18 22:26:32,917 - DEBUG - Debug — pouze konzole


moje_aplikace - DEBUG - Debug — pouze konzole


INFO     | Info — pouze konzole


2026-10-18 22:26:32,918 - INFO - Info — pouze konzole


...

moje_aplikace - WARNING - Warning — konzole + soubor


ERROR    | Error — konzole + soubor


2026-10-18 22:26:32,921 - ERROR - Error — konzole + soubor


moje_aplikace - ERROR - Error — konzole + soubor



✅ Zkontrolujte soubor 'aplikace.log' — najdete tam jen WARNING a ERROR.


---

# Filter — pokročilé filtrování

Kromě úrovně lze zprávy filtrovat i podle **vlastních pravidel** pomocí třídy `logging.Filter` nebo vlastní funkce.

### Příklad — filtr podle názvu loggeru

In [7]:
import logging

# Filtr propustí pouze zprávy z loggerů začínajících na "app.database"
db_filter = logging.Filter("app.database")

handler = logging.StreamHandler()
handler.addFilter(db_filter)
handler.setFormatter(logging.Formatter("%(name)s - %(message)s"))

root = logging.getLogger()
root.addHandler(handler)
root.setLevel(logging.DEBUG)

# Pouze zprávy z app.database projdou filtrem
logging.getLogger("app.database").info("SELECT * FROM users")     # ✅ projde
logging.getLogger("app.auth").info("User logged in")              # ❌ neprojde
logging.getLogger("app.database.pool").info("Connection opened")  # ✅ projde (podřízený)

2026-10-18 22:26:32,927 - INFO - SELECT * FROM users


app.database - INFO - SELECT * FROM users


app.database - SELECT * FROM users


2026-10-18 22:26:32,928 - INFO - User logged in


app.auth - INFO - User logged in


2026-10-18 22:26:32,929 - INFO - Connection opened


app.database.pool - INFO - Connection opened


app.database.pool - Connection opened


### Příklad — vlastní filtr jako funkce

In [8]:
import logging

# Vlastní filtr — propustí jen zprávy obsahující slovo "important"
def only_important(record):
    return "important" in record.getMessage().lower()

logger = logging.getLogger("filtr_demo")
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.addFilter(only_important) #pouze název funkce bez závorek
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

logger.info("This is an important message")   # ✅ projde
logger.info("This is a regular message")       # ❌ neprojde
logger.warning("IMPORTANT: disk almost full")  # ✅ projde

INFO: This is an important message


2026-10-18 22:26:32,934 - INFO - This is an important message


filtr_demo - INFO - This is an important message


2026-10-18 22:26:32,936 - INFO - This is a regular message


filtr_demo - INFO - This is a regular message


2026-10-18 22:26:32,936 - WARNING - IMPORTANT: disk almost full


filtr_demo - WARNING - IMPORTANT: disk almost full


---

# Logování výjimek (exception logging)

Při zachycení výjimky je užitečné zalogovat celý **traceback** (výpis zásobníku). K tomu slouží metoda `logger.exception()` nebo parametr `exc_info=True`.

### Příklad

In [9]:
import logging

logger = logging.getLogger("exception_demo")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

def deleni(a, b):
    try:
        vysledek = a / b
        logger.info(f"{a} / {b} = {vysledek}")
        return vysledek
    except ZeroDivisionError:
        # logger.exception() automaticky zaloguje traceback na úrovni ERROR
        logger.exception(f"Chyba při dělení {a} / {b}")
        return None

deleni(10, 2)  # ✅ OK
deleni(10, 0)  # ❌ Zaloguje chybu + traceback

INFO: 10 / 2 = 5.0


2026-10-18 22:26:32,942 - INFO - 10 / 2 = 5.0


exception_demo - INFO - 10 / 2 = 5.0


ERROR: Chyba při dělení 10 / 0
Traceback (most recent call last):
  File "/tmp/ipykernel_1183/1927018515.py", line 11, in deleni
    vysledek = a / b
               ~~^~~
ZeroDivisionError: division by zero


2026-10-18 22:26:32,948 - ERROR - Chyba při dělení 10 / 0
Traceback (most recent call last):
  File "/tmp/ipykernel_1183/1927018515.py", line 11, in deleni
    vysledek = a / b
               ~~^~~
ZeroDivisionError: division by zero


exception_demo - ERROR - Chyba při dělení 10 / 0
Traceback (most recent call last):
  File "/tmp/ipykernel_1183/1927018515.py", line 11, in deleni
    vysledek = a / b
               ~~^~~
ZeroDivisionError: division by zero


---

# Praktický příklad — logování ve třídě

Ukážeme si, jak vypadá logování v reálnějším scénáři — třída `Hrac` s dvěma loggery (INFO + DEBUG), které zapisují do společného souboru.

### Příklad

In [10]:
import logging

# Společný handler — zápis do souboru
file_handler = logging.FileHandler("logovani.txt", mode="w", encoding="utf-8")
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# Logger pro INFO zprávy (běžné události)
logger_info = logging.getLogger("info_logger")
logger_info.setLevel(logging.INFO)
logger_info.addHandler(file_handler)

# Logger pro DEBUG zprávy (podrobnosti pro vývojáře)
logger_debug = logging.getLogger("debug_logger")
logger_debug.setLevel(logging.DEBUG)
logger_debug.addHandler(file_handler)


class Hrac:
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self.skore = 0
        logger_info.info(f"Hrac {self.jmeno} byl vytvoren.")
        logger_debug.debug(f"Vytvoren objekt Hrac(jmeno='{self.jmeno}', skore={self.skore})")

    def pridat_body(self, body):
        stary_stav = self.skore
        self.skore += body
        logger_info.info(f"Hrac {self.jmeno} ziskal {body} bodu. Celkem ma {self.skore}.")
        logger_debug.debug(f"Zmena skore: {stary_stav} -> {self.skore}")

    def zkontrolovat_vyhru(self, limit):
        if self.skore >= limit:
            logger_info.info(f"Hrac {self.jmeno} vyhral!")
            logger_debug.debug(f"Kontrola vyhry: skore={self.skore}, limit={limit} => VYHRAL")
            return True
        else:
            logger_debug.debug(f"Kontrola vyhry: skore={self.skore}, limit={limit} => NEVYHRAL")
            return False


# Ukázka použití
hrac1 = Hrac("Petr")
hrac1.pridat_body(5)
hrac1.pridat_body(10)
hrac1.zkontrolovat_vyhru(10)

print("✅ Zkontrolujte soubor 'logovani.txt' — najdete tam všechny záznamy.")

2026-10-18 22:26:32,964 - INFO - Hrac Petr byl vytvoren.


info_logger - INFO - Hrac Petr byl vytvoren.


2026-10-18 22:26:32,965 - DEBUG - Vytvoren objekt Hrac(jmeno='Petr', skore=0)


debug_logger - DEBUG - Vytvoren objekt Hrac(jmeno='Petr', skore=0)


2026-10-18 22:26:32,965 - INFO - Hrac Petr ziskal 5 bodu. Celkem ma 5.


...

2026-10-18 22:26:32,968 - INFO - Hrac Petr vyhral!


info_logger - INFO - Hrac Petr vyhral!


2026-10-18 22:26:32,968 - DEBUG - Kontrola vyhry: skore=15, limit=10 => VYHRAL


debug_logger - DEBUG - Kontrola vyhry: skore=15, limit=10 => VYHRAL


✅ Zkontrolujte soubor 'logovani.txt' — najdete tam všechny záznamy.


### Ukázkový obsah souboru `logovani.txt`

```
2025-10-10 09:15:21,362 - info_logger - INFO - Hrac Petr byl vytvoren.
2025-10-10 09:15:21,364 - debug_logger - DEBUG - Vytvoren objekt Hrac(jmeno='Petr', skore=0)
2025-10-10 09:15:21,365 - info_logger - INFO - Hrac Petr ziskal 5 bodu. Celkem ma 5.
2025-10-10 09:15:21,367 - debug_logger - DEBUG - Zmena skore: 0 -> 5
2025-10-10 09:15:21,368 - info_logger - INFO - Hrac Petr ziskal 10 bodu. Celkem ma 15.
2025-10-10 09:15:21,369 - debug_logger - DEBUG - Zmena skore: 5 -> 15
2025-10-10 09:15:21,371 - info_logger - INFO - Hrac Petr vyhral!
2025-10-10 09:15:21,372 - debug_logger - DEBUG - Kontrola vyhry: skore=15, limit=10 => VYHRAL
```

---

# RotatingFileHandler — rotace log souborů

V produkci log soubory neustále rostou. `RotatingFileHandler` automaticky **rotuje** (přejmenuje a vytvoří nový), když soubor překročí zadanou velikost.

```python
from logging.handlers import RotatingFileHandler

handler = RotatingFileHandler(
    "app.log",
    maxBytes=1_000_000,  # 1 MB
    backupCount=5        # uchová 5 starých souborů (app.log.1, app.log.2, …)
)
```

### Příklad

In [11]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("rotace_demo")
logger.setLevel(logging.DEBUG)

# Rotace při 5 KB, max 3 zálohy
handler = RotatingFileHandler(
    "rotace_demo.log",
    maxBytes=5_000,
    backupCount=3,
    encoding="utf-8"
)
handler.setFormatter(logging.Formatter("%(asctime)s - %(message)s"))
logger.addHandler(handler)

# Zapíšeme hodně zpráv, aby se soubor zrotoval
for i in range(50):
    logger.info(f"Zprava cislo {i}: Lorem ipsum dolor sit amet")

print("✅ Ve složce by měly být soubory: rotace_demo.log, .log.1, .log.2, .log.3")

2026-10-18 22:26:32,974 - INFO - Zprava cislo 0: Lorem ipsum dolor sit amet


rotace_demo - INFO - Zprava cislo 0: Lorem ipsum dolor sit amet


2026-10-18 22:26:32,974 - INFO - Zprava cislo 1: Lorem ipsum dolor sit amet


rotace_demo - INFO - Zprava cislo 1: Lorem ipsum dolor sit amet


2026-10-18 22:26:32,975 - INFO - Zprava cislo 2: Lorem ipsum dolor sit amet


...

2026-10-18 22:26:33,005 - INFO - Zprava cislo 48: Lorem ipsum dolor sit amet


rotace_demo - INFO - Zprava cislo 48: Lorem ipsum dolor sit amet


2026-10-18 22:26:33,006 - INFO - Zprava cislo 49: Lorem ipsum dolor sit amet


rotace_demo - INFO - Zprava cislo 49: Lorem ipsum dolor sit amet


✅ Ve složce by měly být soubory: rotace_demo.log, .log.1, .log.2, .log.3


---

# Logování vs. print()

Proč nepoužívat `print()` pro ladění?

| | `print()` | `logging` |
|---|---|---|
| Úrovně závažnosti | ❌ Ne | ✅ DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Výstup do souboru | ❌ Musíte řešit ručně | ✅ FileHandler |
| Formátování (čas, modul…) | ❌ Musíte psát ručně | ✅ Formatter |
| Filtrování | ❌ Nelze | ✅ Level + Filter |
| Vypnutí v produkci | ❌ Musíte mazat/komentovat | ✅ Změna úrovně |
| Více výstupů najednou | ❌ Ne | ✅ Více handlerů |
| Traceback výjimek | ❌ Ručně | ✅ `logger.exception()` |

> **Závěr:** `print()` je pro rychlé ladění. V jakékoli aplikaci, která poběží déle než pár minut, vždy používejte `logging`.

---

# Shrnutí

| Koncept | Co dělá | Příkaz |
|---|---|---|
| Logger | Vytváří zprávy | `logging.getLogger("nazev")` |
| Handler | Posílá zprávy na výstup | `StreamHandler()`, `FileHandler()` |
| Formatter | Formátuje zprávy | `Formatter("%(asctime)s - %(message)s")` |
| Filter | Filtruje zprávy | `Filter("app.database")` |
| Level | Minimální úroveň | `logger.setLevel(logging.WARNING)` |
| Exception | Loguje traceback | `logger.exception("Chyba")` |
| Rotace | Omezuje velikost logu | `RotatingFileHandler(maxBytes=...)` |

### Typická konfigurace pro reálnou aplikaci

```python
import logging
from logging.handlers import RotatingFileHandler

# Vytvoření loggeru
logger = logging.getLogger("moje_app")
logger.setLevel(logging.DEBUG)

# Konzole — DEBUG+ (pro vývoj)
console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))

# Soubor — WARNING+ (pro produkci), rotace 5 MB, 3 zálohy
file = RotatingFileHandler("app.log", maxBytes=5_000_000, backupCount=3)
file.setLevel(logging.WARNING)
file.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))

logger.addHandler(console)
logger.addHandler(file)
```

---

# Cvičení

> V každém cvičení pracujte pouze s modulem `logging` (případně `logging.handlers`). Nemusíte nic instalovat.

## Cvičení 1 — Základní logování

1. Vytvořte logger s názvem `"cviceni1"`.
2. Nastavte úroveň na `DEBUG`.
3. Přidejte `StreamHandler` s formátem: viz `očekávaný výstup`
4. Vypište jednu zprávu na každé úrovni: DEBUG, INFO, WARNING, ERROR, CRITICAL.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
DEBUG: Toto je debug
INFO: Toto je info
WARNING: Toto je warning
ERROR: Toto je error
CRITICAL: Toto je critical
```

</details>

In [12]:
import logging

# 1) Logger vytváří zprávy. DEBUG znamená, že žádnou úroveň nevynecháme.
logger = logging.getLogger("cviceni1")
logger.setLevel(logging.DEBUG)
logger.propagate = False  # Neposílat stejné zprávy ještě nadřazenému loggeru.

# Při opakovaném spuštění buňky odstraníme staré handlery.
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

# 2) StreamHandler vypisuje zprávy do konzole (v Jupyteru pod buňku).
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

# 3) Pět úrovní od nejpodrobnější po nejzávažnější.
logger.debug("Toto je debug")
logger.info("Toto je info")
logger.warning("Toto je warning")
logger.error("Toto je error")
logger.critical("Toto je critical")

DEBUG: Toto je debug


INFO: Toto je info


ERROR: Toto je error


CRITICAL: Toto je critical


**Co si z cvičení 1 zapamatovat:** Logger určuje úroveň zpráv; handler určuje, kam se vypisují; formatter určuje jejich podobu.

## Cvičení 2 — Logování do souboru

1. Vytvořte logger s názvem `"cviceni2"`.
2. Přidejte `FileHandler`, který zapíše logy do souboru `cviceni2.log` (s `mode="w"`).
3. Nastavte formát: viz `očekávaný výstup`
4. Zalogujte 3 zprávy různých úrovní.
5. Na konci otevřete a vypište obsah souboru pomocí `open()` + `read()`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Obsah souboru cviceni2.log:
2025-10-10 12:00:00,000 | INFO | Server started
2025-10-10 12:00:00,001 | WARNING | High memory usage
2025-10-10 12:00:00,002 | ERROR | Database connection lost
```

</details>

In [13]:
import logging

logger = logging.getLogger("cviceni2")
logger.setLevel(logging.DEBUG)
logger.propagate = False
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

# FileHandler ukládá zprávy do souboru; 'w' přepíše starý obsah.
handler = logging.FileHandler("cviceni2.log", mode="w", encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)

logger.info("Server started")
logger.warning("High memory usage")
logger.error("Database connection lost")

# flush zajistí zápis všech zpráv před přečtením souboru.
handler.flush()
with open("cviceni2.log", "r", encoding="utf-8") as soubor:
    print("Obsah souboru cviceni2.log:")
    print(soubor.read(), end="")

# Uzavřením handleru uvolníme soubor.
logger.removeHandler(handler)
handler.close()

Obsah souboru cviceni2.log:
2026-10-18 22:26:33,019 | INFO | Server started
2026-10-18 22:26:33,019 | WARNING | High memory usage
2026-10-18 22:26:33,019 | ERROR | Database connection lost


**Co si z cvičení 2 zapamatovat:** FileHandler ukládá záznamy do souboru. `flush()` zajistí, že je můžete hned číst.

## Cvičení 3 — Dva handlery najednou

1. Vytvořte logger s názvem `"cviceni3"` a úrovní `DEBUG`.
2. Přidejte **dva handlery**:
   - `StreamHandler` s úrovní `DEBUG` (zobrazí vše v konzoli)
   - `FileHandler` s úrovní `ERROR` (do souboru `cviceni3.log` zapíše jen ERROR+)
3. Oba handlery mají mít vlastní formatter.
4. Zalogujte zprávy: DEBUG, INFO, WARNING, ERROR.
5. Ověřte, že v souboru je pouze ERROR.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Konzole ---
DEBUG: Debug zprava
INFO: Info zprava
WARNING: Warning zprava
ERROR: Error zprava

--- Obsah souboru ---
2025-... - cviceni3 - ERROR - Error zprava
```

</details>

In [14]:
import logging

logger = logging.getLogger("cviceni3")
logger.setLevel(logging.DEBUG)
logger.propagate = False
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

# Konzole ukazuje všechny zprávy od DEBUG výše.
console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

# Soubor přijímá pouze zprávy úrovně ERROR a CRITICAL.
file_handler = logging.FileHandler("cviceni3.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.ERROR)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
logger.addHandler(console)
logger.addHandler(file_handler)

print("--- Konzole ---")
logger.debug("Debug zprava")
logger.info("Info zprava")
logger.warning("Warning zprava")
logger.error("Error zprava")
file_handler.flush()

print("\n--- Obsah souboru ---")
with open("cviceni3.log", encoding="utf-8") as soubor:
    obsah = soubor.read()
    print(obsah, end="")
    assert "ERROR - Error zprava" in obsah
    assert "Debug zprava" not in obsah  # Ověření úrovně souborového handleru.

logger.removeHandler(file_handler)
file_handler.close()

DEBUG: Debug zprava


INFO: Info zprava


ERROR: Error zprava


--- Konzole ---

--- Obsah souboru ---
2026-10-18 22:26:33,026 - cviceni3 - ERROR - Error zprava


**Co si z cvičení 3 zapamatovat:** Každý handler může mít vlastní minimální úroveň i vlastní formát.

## Cvičení 4 — Logování výjimek

1. Vytvořte logger `"cviceni4"` s `StreamHandler`.
2. Napište funkci `nacti_cislo(text)`, která:
   - Zkusí převést `text` na `int`
   - Pokud se to podaří, zaloguje `INFO` se zprávou: `"Nacteno cislo: X"`
   - Pokud dojde k `ValueError`, zaloguje chybu pomocí `logger.exception()`
3. Zavolejte funkci s hodnotami `"42"`, `"abc"`, `"100"`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
INFO: Nacteno cislo: 42
ERROR: Chyba pri konverzi: abc
Traceback (most recent call last):
  ...
ValueError: invalid literal for int() with base 10: 'abc'
INFO: Nacteno cislo: 100
```

</details>

In [15]:
import logging

logger = logging.getLogger("cviceni4")
logger.setLevel(logging.DEBUG)
logger.propagate = False
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

def nacti_cislo(text):
    """Vrátí celé číslo, nebo None při neplatném vstupu."""
    try:
        cislo = int(text)  # Např. řetězec '42' převede na číslo 42.
    except ValueError:
        # exception() se volá uvnitř except a vypíše i traceback.
        logger.exception("Chyba pri konverzi: %s", text)
        return None
    else:
        logger.info("Nacteno cislo: %s", cislo)
        return cislo

nacti_cislo("42")
nacti_cislo("abc")  # Záměrně vyvolá ValueError; funkce ji zachytí.
nacti_cislo("100")

INFO: Nacteno cislo: 42


ERROR: Chyba pri konverzi: abc
Traceback (most recent call last):
  File "/tmp/ipykernel_1183/856054639.py", line 17, in nacti_cislo
    cislo = int(text)  # Např. řetězec '42' převede na číslo 42.
ValueError: invalid literal for int() with base 10: 'abc'


INFO: Nacteno cislo: 100


100

**Co si z cvičení 4 zapamatovat:** `try/except` zachytí chybu; `logger.exception()` navíc uloží traceback.

## Cvičení 5 — Kompletní úloha

Vytvořte třídu `BankovniUcet` s atributy `majitel` (str) a `zustatek` (float, výchozí 0).

Metody:
- `vlozit(castka)` — přidá peníze, zaloguje INFO
- `vybrat(castka)` — odečte peníze; pokud není dostatek, zaloguje WARNING a peníze neodečte
- `prevod(cilovy_ucet, castka)` — provede převod, zaloguje INFO; při nedostatku zaloguje ERROR

Konfigurace logování:
- Logger s názvem `"banka"`
- `StreamHandler` (konzole) s úrovní `DEBUG`
- `FileHandler` (`banka.log`) s úrovní `INFO`

Testovací scénář:
1. Vytvořte 2 účty (Alice: 1000, Bob: 500)
2. Alice vloží 500
3. Bob se pokusí vybrat 1000 (nedostatek!)
4. Alice převede 200 na Boba
5. Vypište konečné zůstatky

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
INFO: Ucet vytvoren: Alice, zustatek=1000
INFO: Ucet vytvoren: Bob, zustatek=500
INFO: Alice: vklad 500, novy zustatek=1500
WARNING: Bob: nedostatek prostredku pro vyber 1000 (zustatek=500)
INFO: Prevod 200 z Alice na Bob
DEBUG: Alice: zustatek 1500 -> 1300
DEBUG: Bob: zustatek 500 -> 700

Konecne zustatky: Alice=1300, Bob=700
```

</details>

In [16]:
import logging

# Jediný logger předává zprávy dvěma různým handlerům.
banka_logger = logging.getLogger("banka")
banka_logger.setLevel(logging.DEBUG)
banka_logger.propagate = False
for old_handler in banka_logger.handlers[:]:
    banka_logger.removeHandler(old_handler)
    old_handler.close()

console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

souborovy_handler = logging.FileHandler("banka.log", mode="w", encoding="utf-8")
souborovy_handler.setLevel(logging.INFO)
souborovy_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
banka_logger.addHandler(console)
banka_logger.addHandler(souborovy_handler)

class BankovniUcet:
    def __init__(self, majitel, zustatek=0.0):
        self.majitel = majitel
        self.zustatek = float(zustatek)
        banka_logger.info("Ucet vytvoren: %s, zustatek=%g", self.majitel, self.zustatek)

    def vlozit(self, castka):
        if castka <= 0:
            raise ValueError("Vklad musi byt kladny")
        self.zustatek += castka
        banka_logger.info("%s: vklad %g, novy zustatek=%g", self.majitel, castka, self.zustatek)

    def vybrat(self, castka):
        if castka <= 0:
            raise ValueError("Vyber musi byt kladny")
        if castka > self.zustatek:
            banka_logger.warning("%s: nedostatek prostredku pro vyber %g (zustatek=%g)",
                                 self.majitel, castka, self.zustatek)
            return False
        self.zustatek -= castka
        banka_logger.info("%s: vyber %g, novy zustatek=%g", self.majitel, castka, self.zustatek)
        return True

    def prevod(self, cilovy_ucet, castka):
        if castka <= 0:
            raise ValueError("Prevod musi byt kladny")
        if castka > self.zustatek:
            banka_logger.error("%s: nedostatek prostredku pro prevod %g", self.majitel, castka)
            return False
        stary_muj = self.zustatek
        stary_cil = cilovy_ucet.zustatek
        self.zustatek -= castka
        cilovy_ucet.zustatek += castka
        banka_logger.info("Prevod %g z %s na %s", castka, self.majitel, cilovy_ucet.majitel)
        banka_logger.debug("%s: zustatek %g -> %g", self.majitel, stary_muj, self.zustatek)
        banka_logger.debug("%s: zustatek %g -> %g", cilovy_ucet.majitel, stary_cil, cilovy_ucet.zustatek)
        return True

alice = BankovniUcet("Alice", 1000)
bob = BankovniUcet("Bob", 500)
alice.vlozit(500)
bob.vybrat(1000)  # Neprojde: na účtu je jen 500.
alice.prevod(bob, 200)
print(f"Konecne zustatky: Alice={alice.zustatek:g}, Bob={bob.zustatek:g}")
assert alice.zustatek == 1300 and bob.zustatek == 700
souborovy_handler.flush()
banka_logger.removeHandler(souborovy_handler)
souborovy_handler.close()

INFO: Ucet vytvoren: Alice, zustatek=1000


INFO: Ucet vytvoren: Bob, zustatek=500


INFO: Alice: vklad 500, novy zustatek=1500


INFO: Prevod 200 z Alice na Bob


DEBUG: Alice: zustatek 1500 -> 1300


DEBUG: Bob: zustatek 500 -> 700


Konecne zustatky: Alice=1300, Bob=700


**Co si z cvičení 5 zapamatovat:** Třída uchovává stav účtů, logger monitoruje operace a handlery zapisují na dvě místa.